# 💳 EMIPredict AI - Intelligent Financial Risk Assessment Platform

### Problem Statement & Scope
This project addresses an end-to-end **Dual Machine Learning Problem** for financial risk evaluation across 400,000 applicant records:
1. **Classification Task**: Predicting loan EMI eligibility (`emi_eligibility` into `Eligible`, `High_Risk`, `Not_Eligible`).
2. **Regression Task**: Predicting safe maximum monthly EMI (`max_monthly_emi` in INR).

**Key Domains**: FinTech & Banking, Risk Analytics, Underwriting Automation.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn & Modeling
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from xgboost import XGBClassifier, XGBRegressor
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, root_mean_squared_error, r2_score

# Project modules
sys.path.append(os.path.abspath('../src'))
from feature_engineering import add_derived_features, CATEGORICAL_COLS, NUMERICAL_COLS, ENGINEERED_COLS

print('Environment and libraries imported successfully!')

## 1. Load Processed Dataset
We load the stratified splits generated from the 404,800 records dataset.

In [ ]:
DATA_PATH = '../data/processed/train.parquet'
if not os.path.exists(DATA_PATH):
    DATA_PATH = '../data/sample_data.parquet'

df = pd.read_parquet(DATA_PATH)
print(f'Loaded dataset with shape: {df.shape}')
df.head(3)

## 2. Exploratory Data Analysis (EDA)
Analyzing eligibility distribution and financial correlation.

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
df['emi_eligibility'].value_counts().plot(kind='pie', autopct='%1.1f%%', colors=['#e74c3c', '#2ecc71', '#f39c12'])
plt.title('EMI Eligibility Distribution')
plt.ylabel('')

plt.subplot(1, 2, 2)
sns.countplot(data=df, x='emi_scenario', hue='emi_eligibility', palette='Set2')
plt.title('Eligibility Breakdown across 5 Lending Scenarios')
plt.xticks(rotation=30, ha='right')

plt.tight_layout()
plt.show()

## 3. Feature Engineering Summary
Derived ratios calculated:
- **Debt-to-Income (DTI)**: `current_emi_amount / monthly_salary`
- **Expense-to-Income (ETI)**: `total_monthly_expenses / monthly_salary`
- **Disposable Income**: `monthly_salary - total_obligations`
- **Emergency Cushion**: `emergency_fund / total_obligations`
- **EMI Burden Ratio**: `estimated_requested_emi / disposable_income`

In [ ]:
feature_cols = NUMERICAL_COLS + ENGINEERED_COLS + CATEGORICAL_COLS
print(f'Total input features: {len(feature_cols)}')
print('Top engineered ratios sample:')
df[['dti_ratio', 'expense_to_income_ratio', 'disposable_income', 'emi_burden_ratio']].describe().T

## 4. Model Benchmarks & Results
### Summary of Trained Models:

#### Classification Models (`emi_eligibility`)
| Model | Accuracy (Target >90%) | Macro F1 | ROC-AUC | Status |
|---|---|---|---|---|
| **Logistic Regression (Baseline)** | 90.33% | 0.5965 | 0.9451 | Evaluated |
| **Random Forest Classifier** | 94.35% | 0.6455 | 0.9843 | Evaluated |
| **XGBoost Classifier** | **97.79%** | **0.9081** | **0.9951** | 🏆 **Production Best** |

#### Regression Models (`max_monthly_emi`)
| Model | RMSE (Target < ₹2000) | MAE | R² Score | Status |
|---|---|---|---|---|
| **Ridge Regression** | ₹3,854.21 | ₹2,541.09 | 0.7506 | Evaluated |
| **Random Forest Regressor** | ₹963.41 | ₹326.72 | 0.9844 | Evaluated |
| **XGBoost Regressor** | **₹720.45** | **₹199.76** | **0.9913** | 🏆 **Production Best** |

## 5. Live Production Model Verification
Load the exported best models and test an applicant scenario.

In [ ]:
import joblib

clf_bundle = joblib.load('../models/best_classifier.joblib')
reg_bundle = joblib.load('../models/best_regressor.joblib')

sample_applicant = pd.DataFrame([{
    'age': 32, 'gender': 'Male', 'marital_status': 'Single', 'education': 'Graduate',
    'monthly_salary': 75000.0, 'employment_type': 'Private', 'years_of_employment': 4.5,
    'company_type': 'Mid-size', 'house_type': 'Rented', 'monthly_rent': 12000.0,
    'family_size': 3, 'dependents': 1, 'school_fees': 3000.0, 'college_fees': 0.0,
    'travel_expenses': 4000.0, 'groceries_utilities': 10000.0, 'other_monthly_expenses': 3000.0,
    'existing_loans': 'No', 'current_emi_amount': 0.0, 'credit_score': 740,
    'bank_balance': 180000.0, 'emergency_fund': 120000.0,
    'emi_scenario': 'Personal Loan EMI', 'requested_amount': 250000.0, 'requested_tenure': 24
}])

applicant_feat = add_derived_features(sample_applicant)
clf_pipeline = clf_bundle['pipeline']
label_enc = clf_bundle['label_encoder']
reg_pipeline = reg_bundle['pipeline']

pred_label = label_enc.inverse_transform(clf_pipeline.predict(applicant_feat[feature_cols]))[0]
pred_emi = reg_pipeline.predict(applicant_feat[feature_cols])[0]

print(f'Predicted Eligibility: {pred_label}')
print(f'Predicted Safe Max Monthly EMI: INR {pred_emi:,.2f}')